# Twitter Bot Detection — End-to-End Workflow

This notebook consolidates the former ETL, embedding, exploratory analysis, modeling, and validation notebooks into one reproducible workflow. Run the sections in order. The repository excludes the raw TwiBot-20 files and generated artifacts, so download the data before executing the data-dependent cells.

## Workflow

1. Configure paths and verify the raw data.
2. Extract the legacy JSON data into profile, tweet, domain, neighbor, and label tables.
3. Load the canonical TwiBot-20 files and construct profile features.
4. Generate profile and tweet BERT embeddings when needed.
5. Build graph embeddings, assemble all features, and inspect the data.
6. Select features, train the historical model variants, validate them, and inspect robustness and SHAP explanations.

Install the project first with `pip install -e .`. BERT embedding generation additionally requires `torch` and `transformers`; LightGBM modeling requires `lightgbm`.

In [ ]:
from pathlib import Path
import importlib.util

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.ensemble import RandomForestClassifier

from twitter_bot_detection.eda import profile_data_preprocessing
from twitter_bot_detection.etl import (
    create_id_domain_df,
    create_id_label_df,
    create_id_neighbor_df,
    make_profile_df,
    make_tweets_df,
)
from twitter_bot_detection.feature_selection import (
    backwards_shap_feature_selection,
    fast_metric_with_ci,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Configure data and output locations

The former notebooks used both the original JSON release (`Twi_2020`) and the canonical graph release (`Twibot-20-new-format`). This notebook preserves both inputs: JSON is used for the ETL artifacts and the canonical files are used for graph-aware modeling.

In [ ]:
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

raw_root = repo_root / "datasets" / "Raw"
legacy_raw_dir = raw_root / "Twi_2020"
canonical_raw_dir = raw_root / "Twibot-20-new-format"
processed_dir = repo_root / "datasets" / "Processed"
etl_dir = processed_dir / "ETL"
embedding_dir = processed_dir / "BERT_output"
model_dir = processed_dir / "models"

for directory in (etl_dir, embedding_dir, model_dir):
    directory.mkdir(parents=True, exist_ok=True)

legacy_files = {split: legacy_raw_dir / f"{split}.json" for split in ("train", "test")}
canonical_files = {
    "profile": canonical_raw_dir / "node.part0.parquet",
    "label": canonical_raw_dir / "label.csv",
    "split": canonical_raw_dir / "split.csv",
    "edge": canonical_raw_dir / "edge.csv",
}
has_legacy_data = all(path.exists() for path in legacy_files.values())
has_canonical_data = all(path.exists() for path in canonical_files.values())

pd.Series({"legacy_json": has_legacy_data, "canonical_graph": has_canonical_data})

## 2. Extract the original JSON release

This replaces the first ETL notebook. It creates the profile, tweet, domain, neighbor, and label tables for both train and test files. Skip this section only when the corresponding parquet artifacts already exist.

In [ ]:
if not has_legacy_data:
    print(f"Add train.json and test.json to {legacy_raw_dir} to run legacy ETL.")
else:
    legacy_tables = {}
    extractors = {
        "profile": make_profile_df,
        "tweets": make_tweets_df,
        "domain": create_id_domain_df,
        "neighbor": create_id_neighbor_df,
        "label": create_id_label_df,
    }
    for name, extractor in extractors.items():
        legacy_tables[name] = pd.concat(
            [extractor(path) for path in legacy_files.values()], ignore_index=True
        ).drop_duplicates()
        legacy_tables[name].to_parquet(etl_dir / f"twibot20_{name}.parquet", index=False)

    pd.Series({name: len(table) for name, table in legacy_tables.items()}, name="rows")

## 3. Load the canonical graph data and prepare profile features

The profile preprocessing function joins labels, dataset splits, and graph relationships; computes account tenure and activity ratios; and creates text representations of follow, followed, and friend relationships.

In [ ]:
if not has_canonical_data:
    missing = [str(path.relative_to(repo_root)) for path in canonical_files.values() if not path.exists()]
    raise FileNotFoundError("Download the canonical TwiBot-20 files before continuing: " + ", ".join(missing))

profile_df = pd.read_parquet(canonical_files["profile"])
label_df = pd.read_csv(canonical_files["label"])
split_df = pd.read_csv(canonical_files["split"])
neighbor_df = pd.read_csv(canonical_files["edge"])

for frame in (profile_df, label_df, split_df):
    if "ID" in frame.columns and "id" not in frame.columns:
        frame.rename(columns={"ID": "id"}, inplace=True)

profiles = profile_data_preprocessing(
    profile_df, label_df=label_df, split_df=split_df, neighbor_df=neighbor_df, random_seed=RANDOM_SEED
)
profiles.to_parquet(etl_dir / "preprocessed_profiles.parquet", index=False)
profiles[["id", "label", "split", "tenure", "followers_count", "following_count"]].head()

## 4. Generate BERT text embeddings

The previous profile and tweet embedding notebooks used the `[CLS]` representation from `bert-base-uncased`. Enable this cell to create embeddings for profile text; use the same helper for cleaned tweet text and average the results by user. Keep the generated parquet files to avoid repeating GPU-intensive inference.

In [ ]:
RUN_BERT_EMBEDDINGS = False

def embed_texts(texts, batch_size=128):
    """Return BERT [CLS] embeddings in input order."""
    if importlib.util.find_spec("torch") is None or importlib.util.find_spec("transformers") is None:
        raise ImportError("Install torch and transformers to generate BERT embeddings.")
    import torch
    from transformers import BertModel, BertTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertModel.from_pretrained("bert-base-uncased").to(device).eval()
    values = texts.fillna("").astype(str).tolist()
    batches = []
    with torch.no_grad():
        for start in range(0, len(values), batch_size):
            encoded = tokenizer(values[start:start + batch_size], padding=True, truncation=True, max_length=280, return_tensors="pt")
            encoded = {name: value.to(device) for name, value in encoded.items()}
            batches.append(model(**encoded).last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(batches)

if RUN_BERT_EMBEDDINGS:
    for column in ("description", "name", "screen_name"):
        vectors = pd.DataFrame(embed_texts(profiles[column]), index=profiles["id"])
        vectors.index.name = "id"
        vectors.add_prefix(f"{column}_").reset_index().to_parquet(embedding_dir / f"{column}_embeddings.parquet", index=False)

In [ ]:
# Generate one embedding per user from up to 200 cleaned tweets when legacy ETL has run.
if RUN_BERT_EMBEDDINGS and has_legacy_data:
    tweets = legacy_tables["tweets"].rename(columns={"ID": "id"}).dropna(subset=["tweet"])
    tweets["tweet"] = tweets["tweet"].str.replace(r"@\w+", "", regex=True).str.slice(0, 280)
    tweets = tweets.groupby("id", group_keys=False).head(200)
    tweet_vectors = pd.DataFrame(embed_texts(tweets["tweet"]), index=tweets["id"])
    tweet_embeddings = tweet_vectors.groupby(level=0).mean().add_prefix("tweet_")
    tweet_embeddings.index.name = "id"
    tweet_embeddings.reset_index().to_parquet(embedding_dir / "tweet_embeddings.parquet", index=False)

## 5. Build graph embeddings and assemble the feature table

A bag-of-words representation of follow, followed, and friend IDs followed by TruncatedSVD reproduces the graph-embedding stage. The assembly step combines profile features, graph features, and any previously generated BERT artifacts.

In [ ]:
graph_text = profiles[["follow_string", "followed_string", "friend_string"]].fillna("").agg(" ".join, axis=1)
graph_matrix = CountVectorizer(token_pattern=r"(?u)\b\w+\b", min_df=2).fit_transform(graph_text)
n_graph_components = min(30, graph_matrix.shape[0] - 1, graph_matrix.shape[1] - 1)
if n_graph_components < 1:
    raise ValueError("At least two users and two graph tokens are required for graph embeddings.")
graph_features = pd.DataFrame(
    TruncatedSVD(n_components=n_graph_components, random_state=RANDOM_SEED).fit_transform(graph_matrix),
    columns=[f"graph_{index}" for index in range(n_graph_components)],
)
features = pd.concat([profiles.reset_index(drop=True), graph_features], axis=1)

for artifact in sorted(embedding_dir.glob("*_embeddings.parquet")):
    features = features.merge(pd.read_parquet(artifact), on="id", how="left")

features.to_parquet(processed_dir / "preprocessed_profile_and_text_features.parquet", index=False)
features.shape

## 6. Inspect data and define reproducible splits

The original exploratory notebooks compared feature distributions by bot/human labels. The plot below is a compact equivalent, while the split logic uses the supplied test set and creates a reproducible validation sample from training users.

In [ ]:
features["label"] = pd.to_numeric(features["label"], errors="raise").astype(int)
train_mask = features["split"].astype(str).str.lower().eq("train")
test_mask = features["split"].astype(str).str.lower().eq("test")
val_mask = train_mask & features["random_number"].lt(0.2)
train_mask &= ~val_mask

sns.histplot(data=features, x="tenure", hue="label", stat="density", common_norm=False, bins=30)
plt.title("Account tenure by label")
plt.show()
features.loc[:, ["split", "label"]].value_counts().rename("users")

## 7. Select features and train model variants

The previous modeling notebooks compared text-only, graph/profile, combined, and combined-without-verified variants. The backward SHAP selection call is optional because it is computationally expensive; its log records the selected feature set.

In [ ]:
numeric_features = features.select_dtypes(include=np.number).columns.drop(["label", "random_number"], errors="ignore").tolist()
model_frame = features[numeric_features + ["label"]].replace([np.inf, -np.inf], np.nan)
model_frame[numeric_features] = model_frame[numeric_features].fillna(model_frame.loc[train_mask, numeric_features].median())

variants = {
    "text": [name for name in numeric_features if name.startswith(("description_", "name_", "screen_name_", "tweet_"))],
    "graph_profile": [name for name in numeric_features if name.startswith("graph_") or name in {"tenure", "followers_count", "following_count", "listed_count", "tweet_count"}],
    "combined": numeric_features,
    "combined_without_verified": [name for name in numeric_features if name != "verified"],
}
variants = {name: columns for name, columns in variants.items() if columns}

RUN_FEATURE_SELECTION = False
if RUN_FEATURE_SELECTION:
    selection_log = backwards_shap_feature_selection(
        RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_SEED),
        model_frame.loc[train_mask], model_frame.loc[val_mask], variants["combined"], target="label",
        bootstrap_samples=20, ci_level=0.80, max_iter=10,
    )
    selection_log.to_parquet(processed_dir / "feature_selection_logs.parquet", index=False)
    selection_log.head()

In [ ]:
if importlib.util.find_spec("lightgbm") is None:
    raise ImportError("Install lightgbm to run the historical model-training workflow.")
from lightgbm import LGBMClassifier

models = {}
for name, columns in variants.items():
    model = LGBMClassifier(n_estimators=200, learning_rate=0.01, min_child_samples=5, n_jobs=-1, random_state=RANDOM_SEED)
    model.fit(model_frame.loc[train_mask, columns], model_frame.loc[train_mask, "label"])
    features[f"{name}_score"] = model.predict_proba(model_frame[columns])[:, 1]
    joblib.dump(model, model_dir / f"{name}_model.joblib")
    models[name] = model

features.to_parquet(processed_dir / "final_scored_dataset.parquet", index=False)

## 8. Validate accuracy, robustness, and interpretability

Evaluate every model on train, validation, and test users, include bootstrap AUC confidence intervals, inspect out-of-time performance by account-creation quarter, and use SHAP for feature importance.

In [ ]:
score_columns = [f"{name}_score" for name in models]
split_masks = {"train": train_mask, "validation": val_mask, "test": test_mask}
metrics = []
for split_name, mask in split_masks.items():
    for score in score_columns:
        data = features.loc[mask, ["label", score]].dropna()
        prediction = data[score].ge(0.5)
        metrics.append({
            "split": split_name, "model": score, "auc": roc_auc_score(data["label"], data[score]),
            "average_precision": average_precision_score(data["label"], data[score]),
            "accuracy": accuracy_score(data["label"], prediction), "balanced_accuracy": balanced_accuracy_score(data["label"], prediction),
            "precision": precision_score(data["label"], prediction, zero_division=0), "recall": recall_score(data["label"], prediction, zero_division=0),
            "f1": f1_score(data["label"], prediction, zero_division=0), "mcc": matthews_corrcoef(data["label"], prediction),
        })
metrics_df = pd.DataFrame(metrics)
auc_ci = fast_metric_with_ci(features.loc[mask], predictions=score_columns, target="label", n_samples=100, ci_level=0.95, weight=None)
metrics_df.loc[metrics_df["split"].eq(split_name), ["auc_ci_lower", "auc_ci_upper"]] = auc_ci[["ci_lower", "ci_upper"]].to_numpy()
metrics_df

In [ ]:
features["created_quarter"] = pd.to_datetime(features["created_at"]).dt.to_period("Q").astype(str)
oot_auc = (
    features.loc[test_mask].groupby("created_quarter")
    .apply(lambda data: roc_auc_score(data["label"], data["combined_without_verified_score"]) if data["label"].nunique() == 2 else np.nan)
    .rename("out_of_time_auc")
)
oot_auc.plot(marker="o", title="Out-of-time AUC without verified status")
plt.ylabel("ROC AUC")
plt.show()

import shap
explainer = shap.TreeExplainer(models["combined"])
shap_values = explainer.shap_values(model_frame.loc[test_mask, variants["combined"]])
shap.summary_plot(shap_values[-1] if isinstance(shap_values, list) else shap_values, model_frame.loc[test_mask, variants["combined"]])